<a href="https://colab.research.google.com/github/Sameer-30/PINN_Example/blob/main/Simple_Pendulum_Oscillation_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 1: GENERATE PENDULUM DATA USING RK4
# ══════════════════════════════════════════════════════════════════════════════

def pendulum_rk4(theta0, omega0, g, L, dt, t_end):
    """Solve pendulum ODE using 4th-order Runge-Kutta."""
    n = int(t_end / dt)
    t = np.linspace(0, t_end, n + 1)
    theta = np.zeros(n + 1)
    omega = np.zeros(n + 1)
    theta[0], omega[0] = theta0, omega0

    def f(state):
        """Returns [dθ/dt, dω/dt]"""
        return np.array([state[1], -(g / L) * np.sin(state[0])])

    for i in range(n):
        state = np.array([theta[i], omega[i]])
        k1 = f(state)
        k2 = f(state + 0.5 * dt * k1)
        k3 = f(state + 0.5 * dt * k2)
        k4 = f(state + dt * k3)
        state_new = state + (dt / 6) * (k1 + 2*k2 + 2*k3 + k4)
        theta[i+1], omega[i+1] = state_new

    return t, theta, omega

# --- Parameters ---
g     = 9.81        # gravity (m/s²)
L     = 1.0         # pendulum length (m)
theta0 = 0.5        # initial angle ≈ 28.6° (rad)
omega0 = 0.0        # starts from rest
dt    = 0.01        # time step
t_end = 10.0        # total time

t, theta, omega = pendulum_rk4(theta0, omega0, g, L, dt, t_end)
print(f"Generated {len(t)} points, t = [0, {t_end}]s")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 2: SEQUENTIAL TRAIN/TEST SPLIT
# ══════════════════════════════════════════════════════════════════════════════

t_split = 6.0       # seconds
split_idx = int(t_split / dt)

# Normalize time to [0, 1] based on FULL range
t_norm = (t - t.min()) / (t.max() - t.min())

X_all   = torch.tensor(t_norm, dtype=torch.float32).unsqueeze(1)
Y_all   = torch.tensor(theta,  dtype=torch.float32).unsqueeze(1)

X_train = X_all[:split_idx]
Y_train = Y_all[:split_idx]
X_test  = X_all[split_idx:]
Y_test  = Y_all[split_idx:]

print(f"Train: {len(X_train)} points  (t = 0 to {t_split}s)")
print(f"Test:  {len(X_test)} points  (t = {t_split} to {t_end}s)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 3: DEFINE THE ANN
# ══════════════════════════════════════════════════════════════════════════════

class SimpleANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64),   nn.Tanh(),
            nn.Linear(64, 64),  nn.Tanh(),
            nn.Linear(64, 64),  nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

model = SimpleANN()
print(f"\nModel: 1 → 64 → 64 → 64 → 1  (Tanh)")
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 4: TRAIN
# ══════════════════════════════════════════════════════════════════════════════

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn   = nn.MSELoss()
epochs    = 5000

train_loss_hist = []
test_loss_hist  = []

print(f"\nTraining for {epochs} epochs...")
for ep in range(1, epochs + 1):
    model.train()
    pred = model(X_train)
    loss = loss_fn(pred, Y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Track test loss
    model.eval()
    with torch.no_grad():
        test_loss = loss_fn(model(X_test), Y_test).item()

    train_loss_hist.append(loss.item())
    test_loss_hist.append(test_loss)

    if ep % 1000 == 0 or ep == 1:
        print(f"  Epoch {ep:5d}  |  Train: {loss.item():.6f}  |  Test: {test_loss:.4f}")

print("Done.\n")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 5: PREDICT & EVALUATE
# ══════════════════════════════════════════════════════════════════════════════

model.eval()
with torch.no_grad():
    Y_pred = model(X_all).numpy().flatten()

# Metrics on test region only
theta_test_true = theta[split_idx:]
theta_test_pred = Y_pred[split_idx:]
mse_test  = np.mean((theta_test_true - theta_test_pred)**2)
rmse_test = np.sqrt(mse_test)
ss_res = np.sum((theta_test_true - theta_test_pred)**2)
ss_tot = np.sum((theta_test_true - np.mean(theta_test_true))**2)
r2_test = 1 - ss_res / ss_tot

print(f"Test Region Metrics (t = {t_split}–{t_end}s):")
print(f"  MSE  = {mse_test:.4f}")
print(f"  RMSE = {rmse_test:.4f}")
print(f"  R²   = {r2_test:.4f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 6: PLOT RESULTS
# ══════════════════════════════════════════════════════════════════════════════

# --- Figure 1: Prediction vs Ground Truth ---
fig1, ax1 = plt.subplots(figsize=(8,5))
ax1.axvspan(t_split, t_end, alpha=0.10, color='red', label='Test Region (unseen)')
ax1.plot(t, theta,  color='#1E3A5F', linewidth=0.8, label='RK4 (True)')
ax1.plot(t, Y_pred, color='#DC2626', linewidth=0.8, linestyle='--', label='ANN (Predicted)')
ax1.axvline(x=t_split, color='gray', linewidth=0.5, linestyle=':')
ax1.set_xlabel('Time (s)', fontsize=15)
ax1.set_ylabel('θ (rad)', fontsize=15)
ax1.set_title('ANN Prediction vs Ground Truth', fontsize=9, fontweight='bold')
ax1.legend(fontsize=6, frameon=False)
ax1.tick_params(labelsize=10)
ax1.grid(True, alpha=0.3, linewidth=0.5)
ax1.text(t_split + 0.15, theta.max() * 0.85, f'R² = {r2_test:.3f}',
         fontsize=7, color='#DC2626', fontweight='bold')
fig1.tight_layout()


# --- Figure 2: Loss Curves ---
fig2, ax2 = plt.subplots(figsize=(8, 5))
ax2.semilogy(train_loss_hist, color='#2563EB', linewidth=0.7, label='Train Loss')
ax2.semilogy(test_loss_hist,  color='#DC2626', linewidth=0.7, label='Test Loss')
ax2.set_xlabel('Epoch', fontsize=15)
ax2.set_ylabel('MSE Loss (log)', fontsize=8)
ax2.set_title('Training & Test Loss', fontsize=9, fontweight='bold')
ax2.legend(fontsize=7, frameon=False)
ax2.tick_params(labelsize=7)
ax2.grid(True, alpha=0.3, linewidth=0.5)
fig2.tight_layout()


# --- Figure 3: Error in Test Region ---
fig3, ax3 = plt.subplots(figsize=(8, 5))
t_test_plot = t[split_idx:]
error = theta_test_true - theta_test_pred
ax3.plot(t_test_plot, error, color='#7C3AED', linewidth=0.7)
ax3.fill_between(t_test_plot, error, 0, alpha=0.15, color='#7C3AED')
ax3.axhline(y=0, color='black', linewidth=0.4)
ax3.set_xlabel('Time (s)', fontsize=15)
ax3.set_ylabel('Error: θ_true − θ_pred (rad)', fontsize=8)
ax3.set_title('Extrapolation Error (Test Region)', fontsize=9, fontweight='bold')
ax3.tick_params(labelsize=7)
ax3.grid(True, alpha=0.3, linewidth=0.5)
fig3.tight_layout()

plt.show()
